# LangSmith 기초

LangSmith는 LangChain 애플리케이션의 실행 과정을 기록하고 분석하는 관측성(observability) 플랫폼이다. 이 노트북에서는 Gemini 기반 LCEL 체인을 추적한다.


## 설치 및 환경 변수

필요한 패키지를 설치한다.

```bash
pip install langsmith
```

프로젝트 루트의 `.env`에 다음 값을 설정한다.

```dotenv
GEMINI_API_KEY=...
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=langchain-gemini-prac
```


API 키가 여러 워크스페이스에 연결되어 있다면 `LANGSMITH_WORKSPACE_ID`도 설정한다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT", "default"))

## Gemini 체인 만들기

LangChain의 Runnable을 실행하면 별도의 콜백 코드 없이 LangSmith에 트레이스가 기록된다.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 초보자를 가르치는 Python 튜터야. 핵심만 한국어로 설명해줘."),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
chain = prompt | llm | StrOutputParser()

In [3]:
result = chain.invoke({"question": "Python 데코레이터가 뭐야?"})
print(result)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


안녕하세요! 초보자를 위한 파이썬 튜터입니다. 😊

**데코레이터(Decorator)**는 한 마디로 **"기존 함수를 건드리지 않고, 새로운 기능을 슬쩍 추가(장식)해주는 도구"**입니다.

---

### 💡 쉬운 비유로 이해하기

스마트폰을 하나 샀다고 해볼게요.
* **원래 함수**: 스마트폰 (전화 걸기 기능)
* **데코레이터**: 스마트폰 케이스 (투명 케이스, 거치대 기능 추가)

스마트폰 분해해서 구조를 바꾸지 않아도, 케이스만 씌우면 '거치 기능'이라는 새로운 기능이 추가되죠? 데코레이터가 바로 이 **케이스** 역할을 합니다.

---

### 💻 코드로 보기

함수 위에 **`@데코레이터이름`**을 붙여서 사용합니다.

```python
# 1. 데코레이터(포장지) 만들기
def my_decorator(func):
    def wrapper():
        print("--- [시작] 함수를 실행합니다 ---")  # 추가 기능 1
        func()  # 원래 함수 실행
        print("--- [끝] 함수가 끝났습니다 ---")  # 추가 기능 2

    return wrapper


# 2. 데코레이터 사용하기 (@ 붙이기)
@my_decorator
def say_hello():
    print("안녕하세요!")


# 실행해볼까요?
say_hello()
```

**[실행 결과]**
```text
--- [시작] 함수를 실행합니다 ---
안녕하세요!
--- [끝] 함수가 끝났습니다 ---
```

---

### 🎯 왜 쓸까요? (핵심 장점)

1. **코드 중복 방지**: 여러 함수에 똑같은 기능(예: 실행 시간 측정, 로그인 확인, 로그 남기기)을 넣어야 할 때, 데코레이터 하나만 만들어 두고 `@`만 붙이면 됩니다.
2. **깔끔한 코드**: 원래 함수는 자기 할 일만 집중해서 코드가 깨끗해집니다.

**요약하자면:**  
`@` 표시를 보면 **"아, 이 함수 실행할 때 다른 부가 기능도 같이 실행해

## 태그와 메타데이터 추가하기

`config`를 전달하면 LangSmith 화면에서 실행 목적이나 실험 버전을 쉽게 필터링할 수 있다.

In [4]:
result = chain.invoke(
    {"question": "리스트와 튜플의 차이는 뭐야?"},
    config={
        "run_name": "python-basic-question",
        "tags": ["gemini", "basic", "class-demo"],
        "metadata": {"lesson": "01-1", "model_family": "gemini"},
    },
)
print(result)

안녕하세요! 파이썬에서 **리스트(List)**와 **튜플(Tuple)**의 가장 큰 차이는 **"값을 바꿀 수 있느냐, 없느냐"**입니다.

핵심만 3가지로 정리해 드릴게요!

---

### 1. 수정 가능 여부 (가장 중요!)
* **리스트 (List):** 생성 후에 요소를 **추가, 삭제, 수정할 수 있습니다.** (가변성)
* **튜플 (Tuple):** 한 번 만들면 **절대 값을 바꿀 수 없습니다.** (불변성)

### 2. 괄호 모양
* **리스트:** 대괄호 `[]` 사용 $\rightarrow$ `[1, 2, 3]`
* **튜플:** 소괄호 `()` 사용 $\rightarrow$ `(1, 2, 3)`

### 3. 코드 예시
```python
# --- 리스트 ---
my_list = [1, 2, 3]
my_list[0] = 99  # 가능! -> [99, 2, 3]으로 바뀜

# --- 튜플 ---
my_tuple = (1, 2, 3)
my_tuple[0] = 99  # ❌ 에러 발생! (값을 바꿀 수 없음)
```

---

### 💡 한 줄 요약: 언제 뭘 써야 할까요?
* **리스트:** 데이터가 계속 추가되거나 바뀌어야 할 때 (예: 쇼핑몰 장바구니, 학생 점수 목록)
* **튜플:** 데이터가 절대 변하면 안 되거나, 값이 바뀌는 것을 막고 싶을 때 (예: GPS 위도·경도 좌표, 날짜 정보)

> **팁:** 튜플은 기능이 제한적인 대신, 리스트보다 **속도가 빠르고 메모리를 적게** 사용합니다!


## LangSmith에서 확인하기

[LangSmith](https://smith.langchain.com)에 접속해 `LANGSMITH_PROJECT`에 지정한 프로젝트를 연다.

확인할 항목:

- 체인의 프롬프트 → Gemini 모델 → 출력 파서 실행 순서
- 템플릿 변수가 적용된 최종 프롬프트
- 입력·출력 토큰 수와 각 단계의 소요 시간
- `run_name`, 태그, 메타데이터
- 실패한 실행이 있다면 오류가 발생한 단계

> 환경 변수를 변경했는데 반영되지 않으면 위 환경 설정 셀을 다시 실행한 뒤 커널을 재시작한다.

## 선택적 추적

전체 실행을 항상 기록하지 않고 특정 구간만 추적하려면 `tracing_context`를 사용할 수 있다.

In [ ]:
import langsmith as ls

with ls.tracing_context(enabled=True, project_name="langchain-gemini-selective"):
    result = chain.invoke({"question": "Python 제너레이터가 뭐야?"})

print(result)